In [1]:
import torch
words = open('../names.txt', 'r').read().splitlines()

In [3]:
N = torch.zeros((27, 27), dtype = torch.int32)
chars = sorted(list(set(''.join(words)))) # 把 words 这个字符串列表里的所有单词，拼接成一个大字符串，中间不加任何东西。
stoi = {s:i+1 for i,s in enumerate(chars)}  # stoi = string to integer
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

在 note_2 中，我们每一次预测下一个字母都需要进行一次归一化，这一次我们进行优化，把归一化提前，只计算一次。

首先我们让 P 作为一个 27*27 的矩阵 N，
通过 P.sum(1) 沿着第一维度（也就是每一行）求和，
但是我们会发现，维度只有 1 了，但是只有27维才能使得：27*27 的矩阵 / 27的矩阵 = 每一行的归一值，
所以用到keepdim（保持维度），
但即使是 torch.Size([27, 27]) torch.Size([27, 1])，也还是无法计算（需要求出一行27个数的权重，需要有27个被除数），
所以这里就提到 PyTorch广播了，即：
[
    [1],
    [2],
    [3],
]
变成->
[
    [1,1,1,……],
    [2,2,2,……],
    [3,3,3,……],
]

In [23]:
P = N.float()

In [35]:
print(P)
print(P.sum())
print(P.sum(1))
print(P.sum(1, keepdim = True))
print(P.shape, P.sum(1, keepdim = True).shape)

tensor([[0.0000e+00, 4.4100e+03, 1.3060e+03, 1.5420e+03, 1.6900e+03, 1.5310e+03,
         4.1700e+02, 6.6900e+02, 8.7400e+02, 5.9100e+02, 2.4220e+03, 2.9630e+03,
         1.5720e+03, 2.5380e+03, 1.1460e+03, 3.9400e+02, 5.1500e+02, 9.2000e+01,
         1.6390e+03, 2.0550e+03, 1.3080e+03, 7.8000e+01, 3.7600e+02, 3.0700e+02,
         1.3400e+02, 5.3500e+02, 9.2900e+02],
        [6.6400e+03, 5.5600e+02, 5.4100e+02, 4.7000e+02, 1.0420e+03, 6.9200e+02,
         1.3400e+02, 1.6800e+02, 2.3320e+03, 1.6500e+03, 1.7500e+02, 5.6800e+02,
         2.5280e+03, 1.6340e+03, 5.4380e+03, 6.3000e+01, 8.2000e+01, 6.0000e+01,
         3.2640e+03, 1.1180e+03, 6.8700e+02, 3.8100e+02, 8.3400e+02, 1.6100e+02,
         1.8200e+02, 2.0500e+03, 4.3500e+02],
        [1.1400e+02, 3.2100e+02, 3.8000e+01, 1.0000e+00, 6.5000e+01, 6.5500e+02,
         0.0000e+00, 0.0000e+00, 4.1000e+01, 2.1700e+02, 1.0000e+00, 0.0000e+00,
         1.0300e+02, 0.0000e+00, 4.0000e+00, 1.0500e+02, 0.0000e+00, 0.0000e+00,
         8.4200e+

In [ ]:
P = P / P.sum(1, keepdim=True)
P # 求出了归一化的 tensor对象

tensor([[0.0000e+00, 1.3767e-01, 4.0770e-02, 4.8138e-02, 5.2758e-02, 4.7794e-02,
         1.3018e-02, 2.0885e-02, 2.7284e-02, 1.8450e-02, 7.5610e-02, 9.2498e-02,
         4.9074e-02, 7.9231e-02, 3.5776e-02, 1.2300e-02, 1.6077e-02, 2.8720e-03,
         5.1166e-02, 6.4153e-02, 4.0833e-02, 2.4350e-03, 1.1738e-02, 9.5839e-03,
         4.1832e-03, 1.6702e-02, 2.9001e-02],
        [1.9596e-01, 1.6408e-02, 1.5966e-02, 1.3870e-02, 3.0751e-02, 2.0422e-02,
         3.9546e-03, 4.9579e-03, 6.8821e-02, 4.8694e-02, 5.1645e-03, 1.6763e-02,
         7.4605e-02, 4.8222e-02, 1.6048e-01, 1.8592e-03, 2.4199e-03, 1.7707e-03,
         9.6326e-02, 3.2994e-02, 2.0274e-02, 1.1244e-02, 2.4613e-02, 4.7514e-03,
         5.3711e-03, 6.0499e-02, 1.2838e-02],
        [4.3100e-02, 1.2136e-01, 1.4367e-02, 3.7807e-04, 2.4575e-02, 2.4764e-01,
         0.0000e+00, 0.0000e+00, 1.5501e-02, 8.2042e-02, 3.7807e-04, 0.0000e+00,
         3.8941e-02, 0.0000e+00, 1.5123e-03, 3.9698e-02, 0.0000e+00, 0.0000e+00,
         3.1834e-

In [12]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):
    ix = [0]
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.
ka.
emimmsade.
enkaviyny.
ftlspihinivenvorhlasu.
dsor.
br.
jol.
pen.
aisan.
ja.
